---
title: NCAR S-Pol Radar moments data - Prediction of Rainfall Extremes Campaign In the Pacific (PRECIP) field campaign
author: Ting-Yu Cha, Anna del Moral Mendez, Chia-Wei Hsu
date: "2026-03-25"
---

# Access S-Pol Radar moments- PRECIP through the NCAR GDEX 

:::{important} Radar Data in Analysis Ready Format.
The original data use in this example are archived in the EOL Field Data Archive (https://data.eol.ucar.edu/), under the 'NCAR S-Pol radar moments' folder, as tarred CfRadial1 files, a NetCDF-based format for radial data. The version accessed here is the Analysis/AI-ready data stored in [Zarr format](../docs/zarr_format.md) on NCAR's BOREAS object storage. The only difference between the two sources is the data format. It is accessible both from NCAR HPC and remotely over the internet. 
:::

## Required Packages

Please ensure the following packages are installed before proceeding.

| Category                   | Package      | Purpose |
|---                         |---           |---|
| **Data Access & Analysis** | `zarr`       | Read/write Zarr format data stores |
|                            | `xarray`     | Lazy loading and array manipulation |
| **Radar Analysis**         | `xradar`     | Read and process radar data |
|                            | `arm-pyart`  | ARM radar analysis toolkit |
| **Visualization**          | `matplotlib` | General-purpose plotting |
|                            | `cartopy`    | Geospatial map projections |
|                            | `cmweather`  | Colormaps tailored for weather data |

:::{note} Radar-Specific Packages
`xradar` and `arm-pyart` are specialized libraries for reading and analyzing radar data. For full documentation and advanced usage, refer to their official docs:
- [Xradar Documentation](https://docs.openradarscience.org/projects/xradar/en/stable/)
- [Py-ART Documentation](https://arm-doe.github.io/pyart/)
:::

## Step 1 - Visualize the Radar data
The dataset contains two types of files corresponding to two different S‑Pol scanning strategies: Range–Height Indicator (RHI) and Plan Position Indicator (PPI). In this example, our goal is to quickly plot both types of radar scans and, if desired, combine multiple scans to perform cross‑file analysis.

## Step 2 - Locate the Dataset

The dataset is freely available on the [NCAR GDEX portal](https://gdex.ucar.edu/datasets/d694517/) and is stored on NCAR's BOREAS object storage. This notebook demonstrates two access methods:

- **NCAR HPC** — direct BOREAS access for users on NCAR's HPC systems
- **Remote** — internet-based access for users outside of NCAR


In [ ]:
# Please specify your preferred data access method: the Data URL or the GDEX POSIX path. 
# We are also open to suggestions for the most useful access method for our target users.
hpc_url_horizontal_scan = "https://boreas.hpc.ucar.edu:6443/gdex-data/d694517/v2.0_zarr/cfrad.20220525_030910.927_to_20220525_031137.777_SPOL_PrecipSur2_SUR.zarr"
remote_url_vertical_scan = "https://osdf-director.osg-htc.org/ncar-gdex/d694517/v2.0_zarr/cfrad.20220525_015443.665_to_20220525_015709.985_SPOL_PrecipRhi2_RHI.zarr"

## Step 3 - Open the Data

Radar data in Zarr format is best opened with `xr.open_datatree` rather than the standard `xr.open_dataset`. This is because the data follows the **CfRadial2** convention, which organizes radar sweeps into a hierarchical tree structure — all sweeps from a single scan are stored together in one Zarr store, with each sweep as a separate node in the tree.

For background on the CfRadial2 format and how sweep data is structured, see this article provided by EarthMover: [From Files to Datasets: FM-301 and the Future of Radar Interoperability](https://earthmover.io/blog/from-files-to-datasets-fm-301-and-the-future-of-radar-interoperability)


In [ ]:
import xarray as xr
dt_horizontal = xr.open_datatree(hpc_url_horizontal_scan, engine="zarr", decode_timedelta=False)
dt_vertical = xr.open_datatree(remote_url_vertical_scan, engine="zarr", decode_timedelta=False)

In [ ]:
dt_horizontal

In [ ]:
dt_vertical

## Step 4 - Focus on a Single Sweep

A `DataTree` returned by `xr.open_datatree` contains multiple nodes, each corresponding to one radar sweep (e.g., `sweep_0`, `sweep_1`, ...). To visualize or analyze the data, we first extract a single sweep as a standard `xarray.Dataset` using node indexing.

Each sweep node contains the full set of radar variables (e.g., reflectivity, velocity, differential reflectivity) along with the associated coordinates (range, azimuth or elevation, and time) for that scan.


:::{tip} Extracting a Dataset from a DataTree:

Use `.to_dataset()` on a specific sweep node to convert it into a standard `xarray.Dataset`. Compare the outputs of the `DataTree` cells above with the `Dataset` below — notice how the tree hierarchy collapses into a flat set of variables and coordinates, which is what most plotting and analysis functions expect.
:::

In [ ]:
ds_vertical_sweep0 = dt_vertical["sweep_0"].to_dataset()
ds_vertical_sweep0

## Step 5 - Visualize the RHI Scan

With the vertical sweep extracted, we now prepare and plot the Range-Height Indicator (RHI) scan. The following three cells:

1. **Define a helper function** — computes the altitude (in km) of each radar gate from its range and elevation angle, accounting for Earth's curvature.
2. **Preprocess the sweep** — applies the altitude function to add `rhi_alt` as a new coordinate, converts range from meters to kilometers, and updates variable attributes.
3. **Plot the reflectivity (DBZ)** — renders a 2D `pcolormesh` of reflectivity as a function of range and altitude using the `ChaseSpectral` colormap from `cmweather`.


In [ ]:
import numpy as np
# calculate RHI altitude from range and elevation angle
def calculate_rhi_altitude(r,e):
    """Calculate radar beam altitude (kilometers) from range (meters) and elevation angle (radians).
    """
    rEarth = 6371000.0  # meters
    e_deg = np.deg2rad(e)
    return (np.sqrt(r**2 + rEarth**2 + 2*r*rEarth*np.sin(e_deg)) - rEarth) / 1000.0  # output in km

In [ ]:
# calculate RHI altitude
da_rhi_alt = calculate_rhi_altitude(ds_vertical_sweep0['range'], ds_vertical_sweep0['elevation']).compute()

# add RHI altitude as a coordinate
ds_vertical_sweep0['rhi_alt'] = da_rhi_alt
ds_vertical_sweep0 = ds_vertical_sweep0.set_coords('rhi_alt')

# convert range to km
ds_vertical_sweep0['range'] = ds_vertical_sweep0['range'] / 1000.0  # Convert to km

# update variable attributes
ds_vertical_sweep0['rhi_alt'].attrs = {}
ds_vertical_sweep0['rhi_alt'].attrs['long_name'] = 'RHI Altitude'
ds_vertical_sweep0['rhi_alt'].attrs['units'] = 'kilometers'
ds_vertical_sweep0['range'].attrs['units'] = 'kilometers'

In [ ]:
import matplotlib.pyplot as plt
import cmweather # colorblind friendly weather data visualization (https://cmweather.readthedocs.io/)

# plot DBZ with range and RHI altitude
fig, ax = plt.subplots(figsize=(20, 4),dpi=300)
ds_vertical_sweep0.DBZ.plot.pcolormesh(x='range', y='rhi_alt', cmap="ChaseSpectral", levels=np.arange(-10, 55, 1), ax=ax,add_colorbar=False)
ax.set_xlabel("Range (km)")
ax.set_ylabel("Altitude (km)")
ax.set_title(f"RHI Scan of S-Pol reflectivity (S-Pol location: lon:{ds_vertical_sweep0.longitude.values:0.2f}, lat:{ds_vertical_sweep0.latitude.values:0.2f})", fontsize=16, fontweight='bold')
ax.set_ylim(0,15)
ax.set_xlim(0,150)
plt.colorbar(ax.collections[0], ax=ax, label="Reflectivity (dBZ)")


## Step 6 - Visualize the PPI Scan

This step visualizes the horizontal Plan Position Indicator (PPI) surveillance scan using `xradar` georeferencing utilities. The following three cells:

1. **Georeference the DataTree** — uses `xradar.georeference` to compute Cartesian `x`, `y`, `z` coordinates from the radar's polar coordinates (range, azimuth, elevation), then extracts `sweep_0` as a `Dataset`.
2. **Plot in Cartesian coordinates** — renders the reflectivity field on a flat x/y grid (distance east vs. distance north in meters) using `matplotlib`.
3. **Plot on a geographic map** — reprojects the same reflectivity field onto a `cartopy` map with coastlines and gridlines, providing a geospatial context for the radar coverage area.


In [ ]:
import xradar as xd

### Utilize the xradar georeference utilities to get the x, y, z coordinates for the horizontal scan data.
dt_horizontal = dt_horizontal.xradar.georeference()
dt_horizontal = xd.georeference.get_x_y_z_tree(dt_horizontal)
ds_sweep_0 = dt_horizontal["sweep_0"].to_dataset()
da_dbz = ds_sweep_0["DBZ"]

In [ ]:
#plot reflectivity field on a flat x/y grid 
fig, ax = plt.subplots(figsize=(12, 12),dpi=300)
pcm = da_dbz.plot(
    ax=ax,
    x="x",
    y="y",
    cmap="ChaseSpectral", 
    levels=np.arange(-10, 55, 1), 
    add_colorbar=True,
    cbar_kwargs={
        "label": "Reflectivity (dBZ)",
        "shrink": 0.8,
        "pad": 0.02
    }
)
ax.set_title(f"PPI Scan (Sweep 0) of S-Pol reflectivity (S-Pol location: lon:{da_dbz.longitude.values:0.2f}, lat:{da_dbz.latitude.values:0.2f})", fontsize=16, fontweight='bold')
ax.set_xlabel("Distance East (m)", fontsize=12)
ax.set_ylabel("Distance North (m)", fontsize=12)
ax.set_aspect("equal")  
ax.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
import cartopy
#Plot on geographic map
proj_crs = xd.georeference.get_crs(ds_sweep_0)
cart_crs = cartopy.crs.Projection(proj_crs)

fig = plt.figure(figsize=(12, 12), dpi=300)
ax = fig.add_subplot(111, projection=cartopy.crs.PlateCarree())
da_dbz.plot(
    x="x",
    y="y",
    cmap="ChaseSpectral", 
    levels=np.arange(-10, 55, 1), 
    transform=cart_crs,
    cbar_kwargs=dict(pad=0.075, shrink=0.75),
)
ax.coastlines()
ax.gridlines(draw_labels=True)
ax.set_title(f"PPI Scan of S-Pol reflectivity, Sweep 0 (Elevation angle = {ds_sweep_0.elevation.values[0]:0.2f} deg)", fontsize=16, fontweight='bold')
ax.set_aspect("equal")  
ax.grid(False)
plt.tight_layout()
plt.show()